# CNN-Based Semiconductor Wafer Defect Detection — Kaggle Quickstart

**AI 570 — Team 4** (Paul, Pyle, Rajan, Rettura)

End-to-end training on the WM-811K dataset using a free **Kaggle T4 GPU**.
This is the Kaggle-adapted version of the repo's Colab notebook.

## Before you run (2 clicks, then hit Run All)

Open the notebook's right-hand **Settings** panel and set:

1. **Accelerator** → `GPU T4 x2 (first T4 only used)`
2. **Internet** → `On`

That's it. The dataset is auto-pulled from the team's public Kaggle Dataset
(`brandonpyle/wm-811k-wafer-map`) on first run — no manual `Add Input` step required.

If Kaggle auto-attached the dataset as a notebook input (it usually does
when you open the notebook via the Kaggle website), Cell 3 sees it at
`/kaggle/input/<slug>/LSWMD_new.pkl` and skips the download entirely.
Otherwise it falls back to the Kaggle CLI.

Expected total wall-clock on free P100:
- Cell 1 install: ~30 s
- Cell 3 dataset (download or attach): ~30 s / <1 s
- Cell 5b cache build: ~1 min
- Cell 6 full 3-model run: ~65 min
- Rare-class study (Cell 9): ~30 min

Well inside Kaggle's 9-hour weekly GPU quota and the 12-hour session cap.

---

## Cell 1. Clone repo and install

In [ ]:
%%time
REPO_URL = 'https://github.com/bpyle02/CNN-Based-Semiconductor-Wafer-Defect-Detection-Using-the-WM-811K-Dataset.git'
REPO_DIR = '/kaggle/working/wafer'
BRANCH   = 'main'

import os, subprocess, sys
from pathlib import Path

def _run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

if Path(REPO_DIR).exists():
    print(f'Repo already cloned at {REPO_DIR}; syncing with origin/{BRANCH}...')
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--quiet', 'origin', BRANCH])
    # Stash any local modifications (Kaggle-written checkpoints, metrics, etc.)
    # so the fast-forward pull can never fail. We pop afterward to restore them.
    status = _run(['git', '-C', REPO_DIR, 'status', '--porcelain'])
    has_local = bool(status.stdout.strip())
    if has_local:
        print('Local modifications detected; stashing before pull:')
        print(status.stdout)
        subprocess.check_call(['git', '-C', REPO_DIR, 'stash', 'push',
                               '--include-untracked', '-m', 'kaggle-auto-stash'])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH])
    if has_local:
        pop = _run(['git', '-C', REPO_DIR, 'stash', 'pop'])
        if pop.returncode != 0:
            print('Stash pop had conflicts - your local changes are preserved in `git stash list`.')
            print(pop.stdout); print(pop.stderr)
        else:
            print('Restored local modifications.')
    head = _run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'])
    print(f'HEAD: {head.stdout.strip()}')
else:
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1',
                            '--single-branch', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print('Current working directory:', os.getcwd())

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'])
print('Install complete.')


## Cell 2. GPU + environment snapshot

Prints everything that typically causes "works on my machine" drift — Python
version, torch/CUDA, environment variables, and the GPU specs Kaggle handed
us. Worth reading once so you know what hardware your numbers came from.

In [ ]:
import os, sys, platform, torch, subprocess

print(f'Python:          {sys.version.split()[0]} ({platform.platform()})')
print(f'torch:           {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version:    {torch.version.cuda}')
    print(f'cuDNN version:   {torch.backends.cudnn.version()}')
    print(f'GPU count:       {torch.cuda.device_count()}')
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  [{_i}] {_p.name}  {_p.total_memory/1e9:.1f} GB')
else:
    print('NO GPU — open Settings on the right and set Accelerator = GPU T4 x2 (first T4 only used).')

for _var in ['CUDA_VISIBLE_DEVICES', 'KAGGLE_KERNEL_RUN_TYPE',
             'KAGGLE_DOCKER_IMAGE', 'PYTHONPATH']:
    print(f'env {_var!r}: {os.environ.get(_var, "<unset>")}')

print()
try:
    _out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.used,memory.total,driver_version',
         '--format=csv'], text=True)
    print(_out)
except Exception:
    print('nvidia-smi not available (CPU runtime?).')


## Cell 3. Locate the WM-811K dataset

Finds `LSWMD_new.pkl` in any `/kaggle/input/<slug>/` directory and symlinks
it to `data/LSWMD_new.pkl` so the rest of the pipeline (which expects the
repo-relative path) works unchanged. Symlinking is free; no 2.2 GB copy.

**If this cell errors with "no pkl found"**: attach the dataset via
Settings → Input → Add Input, then re-run.

In [ ]:
import os, glob, subprocess, sys
from pathlib import Path

DATASET_SLUG = 'brandonpyle/wm-811k-wafer-map'
DATASET_TARGET = Path('data/LSWMD_new.pkl')
DATASET_TARGET.parent.mkdir(exist_ok=True, parents=True)


def size_gb(p):
    return os.path.getsize(p) / 1e9


def pick_pkl(paths):
    """From a list of pkl paths, pick the first that looks like a full
    WM-811K wafer map dump (accept either LSWMD_new.pkl or LSWMD.pkl,
    must be > 1 GB)."""
    for p in paths:
        if os.path.basename(p) in ("LSWMD_new.pkl", "LSWMD.pkl") and size_gb(p) > 1.0:
            return p
    return None


# 1. Already linked from a prior run.
if DATASET_TARGET.exists() and size_gb(DATASET_TARGET) > 1.0:
    print(f'Dataset already at {DATASET_TARGET} ({size_gb(DATASET_TARGET):.2f} GB)')

else:
    src = None

    # 2. Kaggle may have auto-attached our dataset as an input (happens if
    #    you open the notebook via the Kaggle website's "Copy & Edit").
    input_candidates = glob.glob('/kaggle/input/**/*.pkl', recursive=True)
    src = pick_pkl(input_candidates)
    if src:
        print(f'Found attached at {src} ({size_gb(src):.2f} GB)')

    # 3. Not attached. Pull via the Kaggle CLI. Kaggle notebooks preload
    #    the running user's credentials so no kaggle.json is needed.
    if src is None:
        print(f'Dataset not attached; downloading {DATASET_SLUG} via kaggle CLI...')
        dl_dir = Path('/kaggle/working/wm811k_dl')
        dl_dir.mkdir(exist_ok=True, parents=True)
        try:
            subprocess.check_call([
                'kaggle', 'datasets', 'download', '-d', DATASET_SLUG,
                '-p', str(dl_dir), '--unzip', '--quiet',
            ])
        except FileNotFoundError:
            # Kaggle CLI not on PATH — install it, retry.
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])
            subprocess.check_call([
                'kaggle', 'datasets', 'download', '-d', DATASET_SLUG,
                '-p', str(dl_dir), '--unzip', '--quiet',
            ])
        dl_candidates = glob.glob(str(dl_dir / '**/*.pkl'), recursive=True)
        src = pick_pkl(dl_candidates)
        if src:
            print(f'Downloaded to {src} ({size_gb(src):.2f} GB)')

    if src is None:
        raise RuntimeError(
            f'Could not locate LSWMD_new.pkl or LSWMD.pkl. Verify Internet '
            f'is ON in notebook Settings, then re-run. If the Kaggle dataset '
            f'{DATASET_SLUG} is private to you, log in under the same '
            f'account in Settings -> Account.'
        )

    # Symlink so the pipeline's default 'data/LSWMD_new.pkl' works without
    # changing any config or CLI flag. The load_dataset() helper accepts
    # either LSWMD_new.pkl or LSWMD.pkl content (it derives failureClass
    # from failureType on load), so linking either is safe.
    if DATASET_TARGET.exists() or DATASET_TARGET.is_symlink():
        DATASET_TARGET.unlink()
    DATASET_TARGET.symlink_to(src)
    print(f'Linked {DATASET_TARGET} -> {src}')

print(f'Ready: {DATASET_TARGET} ({size_gb(DATASET_TARGET):.2f} GB)')


## Cell 4. Sanity-check the dataset

In [ ]:
%%time
# Subprocess so the dataframe doesn't stick around in the notebook kernel.
!python -c "from src.data import load_dataset; df = load_dataset('data/LSWMD_new.pkl'); print('shape:', df.shape); print(df['failureClass'].value_counts())"


## Cell 5. Run the test suite (~1-2 min on T4)

In [ ]:
%%time
!pytest -q --maxfail=5


## Cell 5b. Pre-resize the dataset once (~1 min on Kaggle)

Builds `data/LSWMD_cache.npz` (1.3 MB sidecar) + `data/LSWMD_cache.maps.npy`
(3.19 GB memmap). Peak RSS during build ~1.5 GB so there's no OOM risk on
Kaggle's 13 GB kernel.

In [ ]:
%%time
!python scripts/precompute_tensors.py


## Cell 5c. 1-epoch preflight (~1-2 min on T4)

One fast epoch to confirm the end-to-end pipeline works before committing
to the full 10-epoch multi-model run in Cell 6.

In [ ]:
%%time
# Ensure cache is present before the 1-epoch smoke — identical guard as
# Cell 6 below.
import sys, subprocess
from pathlib import Path
_cache = Path('data/LSWMD_cache.npz')
if not _cache.exists():
    print('Cache missing; building via scripts/precompute_tensors.py...', flush=True)
    _rc = subprocess.call([sys.executable, 'scripts/precompute_tensors.py'])
    if _rc != 0 or not _cache.exists():
        raise RuntimeError(f'Cache build failed (exit {_rc}); see output above.')
else:
    print(f'Cache present: {_cache}')

!python train.py --model cnn --epochs 1 --batch-size 32 --device cuda --seed 42 2>&1 | tail -30


## Cell 6. Train all three models on T4

Expected P100 wall-clock (with AMP + persistent_workers + cudnn.benchmark
which are all enabled by default post-`83ba643`):

| Model | Time | Macro F1 (target) |
|---|---:|---:|
| Custom CNN | ~25 min | 0.79 |
| ResNet-18 | ~18 min | 0.70-0.80 (with new defaults) |
| EfficientNet-B0 | ~20 min | 0.70-0.80 (with new defaults) |

~65 min total. Runs each model as its own subprocess so RAM is released
cleanly between models.

In [ ]:
%%time
EPOCHS = 10
BATCH_SIZE = 64
SEED = 42

import os, gc, subprocess, sys, time, torch
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Preflight: cache build (single-process, memory-safe). Fast if Cell 5b
# already ran.
_cache = Path('data/LSWMD_cache.npz')
if not _cache.exists():
    print('Cache not found; building via scripts/precompute_tensors.py (~1 min)...', flush=True)
    _rc = subprocess.call([sys.executable, 'scripts/precompute_tensors.py'])
    if _rc != 0 or not _cache.exists():
        raise RuntimeError(f'Cache build failed (exit {_rc}); check output above.')
else:
    print(f'Cache present: {_cache} ({_cache.stat().st_size/1e6:.1f} MB sidecar)')

# RAM + GPU baseline
import psutil
_vm = psutil.virtual_memory()
print(f'Kernel RAM before training: {_vm.used/1e9:.1f} / {_vm.total/1e9:.1f} GB used')
if torch.cuda.is_available():
    _gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {_gpu.name} ({_gpu.total_memory/1e9:.1f} GB)')
else:
    print('WARNING: CUDA not available; training will be very slow.')

Path('results').mkdir(exist_ok=True)

_failures = []
_t_overall = time.time()
for _model in ['cnn', 'resnet', 'efficientnet']:
    print(f'\n===== training {_model} =====', flush=True)
    _t0 = time.time()
    _cmd = [sys.executable, 'train.py', '--model', _model,
            '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
            '--device', 'cuda', '--seed', str(SEED)]

    _log_path = f'results/train_{_model}.log'
    with open(_log_path, 'w') as _logf:
        _proc = subprocess.Popen(_cmd, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, bufsize=1,
                                  universal_newlines=True)
        for _line in _proc.stdout:
            sys.stdout.write(_line)
            sys.stdout.flush()
            _logf.write(_line)
        _rc = _proc.wait()

    _elapsed = time.time() - _t0
    if _rc == 0:
        print(f'\n[{_model}] OK ({_elapsed/60:.1f} min) -- log: {_log_path}')
    else:
        print(f'\n[{_model}] FAILED with exit code {_rc} after {_elapsed/60:.1f} min')
        _failures.append(_model)
    for _ in range(3):
        gc.collect()
    torch.cuda.empty_cache()

print(f'\nTotal wall-clock: {(time.time()-_t_overall)/60:.1f} min')
if _failures:
    print(f'\n!!! FAILED MODELS: {_failures} -- cat results/train_<model>.log for details')
else:
    print('\nAll three models trained successfully.')


## Cell 7. Inspect results

In [ ]:
import json
from pathlib import Path

metrics_path = Path('results/metrics.json')
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print(f'Models in metrics.json: {list(metrics.keys())}\n')
    for name, entry in metrics.items():
        print(f'--- {name} ---')
        print(f'  accuracy    : {entry.get("accuracy", "?"):.4f}')
        print(f'  macro F1    : {entry.get("macro_f1", "?"):.4f}')
        print(f'  weighted F1 : {entry.get("weighted_f1", "?"):.4f}')
        print(f'  ECE         : {entry.get("ece", "?"):.4f}')
        print(f'  time (s)    : {entry.get("time_sec", "?"):.1f}')
        print()
else:
    print('No metrics.json yet. Run Cell 6 first.')


In [ ]:
from IPython.display import Image, display
import glob

imgs = sorted(glob.glob('results/*.png'))
if imgs:
    for p in imgs:
        print(p)
        display(Image(p))
else:
    print('No result images yet (they are generated inside train.py after each model).')


## Cell 8. Save outputs to the Kaggle notebook's output

Files under `/kaggle/working` persist as this notebook's attached output and
are downloadable from the "Output" tab on the notebook page. Copy the
canonical artifacts there (by default we run from
`/kaggle/working/wafer/` so they're already under `/kaggle/working`, but an
explicit copy makes them easier to find).

In [ ]:
import shutil
from pathlib import Path

STAGING = Path('/kaggle/working/outputs')
STAGING.mkdir(exist_ok=True, parents=True)

# Copy (not move — keep them in the repo dir for downstream cells).
for src in [Path('results/metrics.json'),
            *Path('results').glob('*.png'),
            *Path('results').glob('*.log'),
            *Path('checkpoints').glob('*.pth')]:
    if src.exists():
        dst = STAGING / src.name
        shutil.copy2(src, dst)
        print(f'  {src} -> {dst}')

print(f'\nStaged {sum(1 for _ in STAGING.iterdir())} files under {STAGING}')
print('Download them from the notebook Output tab after running.')


## Cell 9. Rare-class study (~30-40 min on T4, optional)

Runs the custom CNN under five rebalancing interventions
(baseline, focal loss, weighted CE + DRW, class-balanced sampling,
synthetic augmentation) × three seeds = 15 runs. Results and per-class
recall on rare classes are written to `docs/rare_class_study.md`.

Resumable: skips already-completed (condition, seed) pairs on rerun.
This will eat ~30 min of your weekly GPU quota; skip if you just want
the baseline.

In [ ]:
%%time
!python scripts/run_rare_class_study.py \
    --epochs 10 --batch-size 64 --device cuda \
    --seeds 0,1,2 --conditions all

from IPython.display import Markdown, display
from pathlib import Path
_path = Path('docs/rare_class_study.md')
if _path.exists():
    display(Markdown(_path.read_text(encoding='utf-8')))
else:
    print('Study did not complete; docs/rare_class_study.md was not written.')


## Cell 10. SimCLR bonus (~20-25 min on T4, optional)

Runs only after Cell 9 has written `docs/rare_class_study.md`. SimCLR
pretraining + CE fine-tune on 3 seeds, appended as a bonus section.

In [ ]:
%%time
!python scripts/run_simclr_bonus.py \
    --pretrain-epochs 5 --finetune-epochs 10 \
    --batch-size 64 --device cuda \
    --seeds 0,1,2

from IPython.display import Markdown, display
from pathlib import Path
_path = Path('docs/rare_class_study.md')
if _path.exists():
    display(Markdown(_path.read_text(encoding='utf-8')))


---

**Done.** Inspect the confusion matrices and per-class F1 scores — macro-F1
is the coursework target.

- Model checkpoints live under `/kaggle/working/wafer/checkpoints/`
- Metrics live in `/kaggle/working/wafer/results/metrics.json`
- Training logs per model: `/kaggle/working/wafer/results/train_<model>.log`
- All of the above are also copied to `/kaggle/working/outputs/` by Cell 8.

The notebook's "Output" tab on the right-hand sidebar shows everything
written under `/kaggle/working` and lets you download individual files or
the whole tree as a zip.


## Cell 11. Power architectures: RIDE + Swin (~35 min on T4)

RIDE (long-tail expert ensemble, Wang et al. 2021) is the single strongest
architecture on extreme class imbalance — exactly the WM-811K regime.
Swin-Tiny brings shift-window self-attention that captures spatial wafer
patterns better than plain CNN backbones. Both get DRW loss + synthetic
augmentation via the `C_drw` + `--synthetic` flag combination that won
the rare-class study.

Trains two additional models and saves `checkpoints/ride_best.pth` and
`checkpoints/swin_best.pth` for the ensemble evaluation in Cell 12.


In [ ]:
%%time
import os, subprocess, sys, time, gc, torch

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

_power_models = [
    # (model_name, batch_size, extra_flags)
    ('ride', 128, ['--synthetic', '--config', 'config.yaml',
                   '--config', 'configs/rare_class/C_drw.yaml']),
    ('swin', 64,  ['--synthetic', '--config', 'config.yaml',
                   '--config', 'configs/rare_class/C_drw.yaml']),
]

_failures, _t_all = [], time.time()
for _model, _bs, _extra in _power_models:
    print(f'\n===== training {_model} (batch_size={_bs}) =====', flush=True)
    _t0 = time.time()
    _cmd = [sys.executable, 'train.py', '--model', _model,
            '--epochs', '20', '--batch-size', str(_bs),
            '--device', 'cuda', '--seed', '42'] + _extra
    _log = f'results/train_{_model}.log'
    with open(_log, 'w') as _f:
        _p = subprocess.Popen(_cmd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, bufsize=1,
                              universal_newlines=True)
        for _line in _p.stdout:
            sys.stdout.write(_line); sys.stdout.flush(); _f.write(_line)
        _rc = _p.wait()
    _el = (time.time() - _t0) / 60
    if _rc == 0:
        print(f'[{_model}] OK ({_el:.1f} min) -- log: {_log}')
    else:
        print(f'[{_model}] FAILED exit={_rc} after {_el:.1f} min')
        _failures.append(_model)
    for _ in range(3): gc.collect()
    torch.cuda.empty_cache()

print(f'\nPower-arch total: {(time.time()-_t_all)/60:.1f} min')
if _failures:
    print(f'FAILED: {_failures}')


## Cell 12. Ensemble evaluation (~3 min)

Auto-discovers every `checkpoints/*_best.pth`, reproduces the deterministic
test split (seed=42, 15%), and evaluates:

- Each model individually
- **Averaging**: mean of softmax probabilities (strong default)
- **Voting**: majority argmax vote
- **Weighted**: per-model weights fit on val split to maximize Macro F1

Typically the weighted ensemble beats the best single model by 3–5 Macro F1
points. Results written to `results/ensemble_metrics.json` and rendered as
a markdown table at `docs/ensemble_results.md`.


In [ ]:
%%time
!python scripts/evaluate_ensemble.py --batch-size 128 --device cuda

from IPython.display import Markdown, display
from pathlib import Path
_p = Path('docs/ensemble_results.md')
if _p.exists():
    display(Markdown(_p.read_text(encoding='utf-8')))
else:
    print('Ensemble evaluation did not complete.')
